# MIDI 2026 — Notebook 18 v2: Comprehensive CORE Feature Ablation

**Amaç:** Hakemin istediği `− self_consistency` ve `− unknown_response` analizlerini **mutlaka** gerçekleştirirken, tüm sekiz CORE özelliğinin tek tek çıkarıldığı **leave-one-feature-out (LOFO)** deneylerini ve **Internal-only / Behavioral-only** gruplarını karşılaştırmak.

### Özellik aileleri (bu notebook'ta operasyonel ayrım)

- **Internal + output metadata** (4): `mean_logprob`, `min_logprob`, `mean_token_entropy_normalized`, `output_content_token_count`. İlk üçü token olasılık/entropi sinyali; **çıktı uzunluğu, doğrudan bir içsel olasılık değil, üretim metadata'sıdır**.
- **Behavioral / response-derived** (4): `self_confidence_scaled`, `verification_p_correct`, `self_consistency`, `unknown_response`. Son özellik bir yanıt göstergesidir; kendi başına ek bir LLM çağrısı gerektirmez. **Önemli:** `verification_p_correct`, bir self-verification değerlendirmesinin next-token olasılıklarından hesaplandığı için bu gruplama *salt black-box erişim* anlamına gelmez.

### Sabit deney ailesi

- **Full CORE (8)**: Stage14 dondurulmuş Logistic CORE ve XGBoost CORE tahminleri, yeniden eğitim yok.
- **8 adet LOFO (7 özellik)**: her CORE özelliği tek tek çıkarılır. `minus_consistency` ve `minus_unknown` hakemin talepleridir.
- **`minus_both` (6)**: hem consistency hem unknown çıkarılır; hakem karşılaştırmasını tamamlayan birleşik ablation.
- **`internal_only` (4)** ve **`behavioral_only` (4)**: sinyal ailelerinin tek başına performansı.

Toplam: **11 yeni özellik yapılandırması × 2 kalibratör × 3 LLM = 66 yeni küçük kalibratör**, **16.500 yeni tahmin**, **0 yeni LLM inference**. Orijinal 6 CORE kalibratörü yeniden eğitilmez. Tüm sonuçlar aynı 970 Clean-Dev/model eğitim ve 250 final-test/model test sorusunda hesaplanır.

### Metodolojik ve veri güvenliği kuralları

1. Test etiketleri, yeni tahminlerin dondurulmasından ve manifest oluşturulmasından **sonra** açılır. Test üzerinde hiperparametre seçimi yapılmaz.
2. Logistic Regression ve XGBoost'un Stage12/14 sabit parametreleri kullanılır. Bu nedenle bu deneyler **post-review exploratory** statüsündedir, orijinal önkayıtlı birincil testin parçası olarak sunulmaz.
3. Tüm eski Stage08–17 kaynakları salt okunurdur. Bu v2 sürümü yalnızca `results/midi2026_revision/v1/02_feature_ablation/` ve `.../manifests/` altına **`_v2` adlı ayrı çıktılar** yazar. Notebook 18 v1 sonuçları varsa onları değiştirmez.
4. Brier farkı: **Δ = Brier(alternative subset) − Brier(full CORE)**. Pozitif değer Full CORE lehine, negatif değer özellik alt kümesi lehinedir. Bu farklar koşullu **model yeniden-eğitimi etkisidir; nedensel özellik önemi değildir**.
5. Eşleştirilmiş %95 bootstrap aralıkları **betimleyici ve çoklu karşılaştırma için düzeltilmemiştir**; 66 karşılaştırmada önemli bulgular ihtiyatla yorumlanmalıdır.
6. Heatmap **Δ Brier**'ı bütün modeller ve kalibratörler için gösterir; ayrıca sayısal tablo + publication-ready PNG/PDF kaydeder.
7. Hesaplama maliyeti hakkında gerçek GPU süresi veya token tasarrufu ölçülmüş gibi bir iddia oluşturulmaz. `self_consistency` 5 yeniden üretim gerektirir; tek özellik çıkarmak, başka özellikler aynı çıktıları gerektiriyorsa mutlaka 5 çağrı tasarruf edildiği anlamına gelmez.


In [ ]:
# 1 — Colab/Drive, orijinal protokol sabitleri ve çıktı yolları
import sys, subprocess, importlib.util
for module,pkg in [('pyarrow','pyarrow>=15,<25'),('sklearn','scikit-learn>=1.5,<1.9'),('xgboost','xgboost>=2,<4')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

from pathlib import Path
import os, json, hashlib, math, platform
import numpy as np
import pandas as pd
import sklearn, xgboost
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from io import BytesIO
from xgboost import XGBClassifier

INPUT_MODE='drive'  # 'drive' (Colab) veya 'local' (önceden yerleştirilmiş aynı klasör yapısı)
if INPUT_MODE=='drive':
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/LLM_Confidence_Thesis')
elif INPUT_MODE=='local':
    ROOT=Path('/content/LLM_Confidence_Thesis')
else:
    raise ValueError('INPUT_MODE drive/local olmalı')
assert ROOT.is_dir(),f'Proje klasörü yok: {ROOT}'

DEV='validated_v007'; TEST='validated_v012_final250_v1'
REV=ROOT/'results'/'midi2026_revision'/'v1'
OUT=REV/'02_feature_ablation'; MAN=REV/'manifests'
CORE=['mean_logprob','min_logprob','mean_token_entropy_normalized',
      'output_content_token_count','self_confidence_scaled','verification_p_correct',
      'self_consistency','unknown_response']
# Sekiz CORE özelliğinin mantıksal grupları
INTERNAL=CORE[:4]       # Token-level features + response-length metadata
BEHAVIORAL=CORE[4:]     # Self-report, verification, resampling, response marker
REVIEWER_FEATURES=['self_consistency','unknown_response']
LEAVE_ONE_OUT={
 'minus_mean_logprob':'mean_logprob',
 'minus_min_logprob':'min_logprob',
 'minus_entropy':'mean_token_entropy_normalized',
 'minus_output_length':'output_content_token_count',
 'minus_self_confidence':'self_confidence_scaled',
 'minus_verification':'verification_p_correct',
 'minus_consistency':'self_consistency',
 'minus_unknown':'unknown_response',
}
FEATURES={name:[v for v in CORE if v!=removed] for name,removed in LEAVE_ONE_OUT.items()}
FEATURES['minus_both']=[v for v in CORE if v not in REVIEWER_FEATURES]
FEATURES['internal_only']=INTERNAL.copy()
FEATURES['behavioral_only']=BEHAVIORAL.copy()
VARIANT_ORDER=list(FEATURES)
VARIANT_LABELS={
 'minus_mean_logprob':'− Mean log-prob',
 'minus_min_logprob':'− Min log-prob',
 'minus_entropy':'− Token entropy',
 'minus_output_length':'− Output length',
 'minus_self_confidence':'− Self-confidence',
 'minus_verification':'− Self-verification',
 'minus_consistency':'− Self-consistency',
 'minus_unknown':'− Unknown indicator',
 'minus_both':'− Consistency & unknown',
 'internal_only':'Internal (+ length) only',
 'behavioral_only':'Behavioral only',
}
N_VARIANTS=11
N_FITS=3*2*N_VARIANTS
N_NEW_PREDICTIONS=N_FITS*250
N_METHODS=2*(N_VARIANTS+1) # CORE + subsets per family
assert len(FEATURES)==N_VARIANTS and len(set(tuple(v) for v in FEATURES.values()))==N_VARIANTS
assert all(v and set(v).issubset(CORE) for v in FEATURES.values())
assert len(INTERNAL)==4 and len(BEHAVIORAL)==4 and set(INTERNAL).isdisjoint(BEHAVIORAL)
assert set(INTERNAL+BEHAVIORAL)==set(CORE)

KEY=['model_id','question_id']; TARGET='manual_is_correct'
BASE_METHODS=['logistic_core','xgb_core']
METHODS=[f'{alg}_{ab}' for alg in ['logistic','xgb'] for ab in FEATURES]
BOOT_B=4000;BOOT_SEED=20260922;EPS=1e-6;ECE_BINS=10

P={
 'dev_extended':ROOT/'features'/'assembled'/DEV/'extended_model_matrix_v2_repair1.parquet',
 'dev_schema':ROOT/'features'/'assembled'/DEV/'feature_schema_v1.json',
 'dev_manifest':ROOT/'features'/'assembled'/DEV/'stage08_manifest_v2_repair1.json',
 'test_core':ROOT/'features'/'assembled'/TEST/'core_model_matrix_v1.parquet',
 'test_manifest':ROOT/'features'/'assembled'/TEST/'stage08_manifest_v1.json',
 'stage14_predictions':ROOT/'results'/'stage14'/TEST/'final_test_predictions_v1.parquet',
 'stage14_pred_manifest':ROOT/'results'/'stage14'/TEST/'stage14_predictions_manifest_v1.json',
 'stage14_metrics':ROOT/'results'/'stage14'/TEST/'final_test_metrics_v1.csv',
 'stage12_protocol':ROOT/'results'/'stage12'/DEV/'final_test_protocol_v2.json',
 'stage14_fit_audit':ROOT/'results'/'stage14'/TEST/'development_fit_audit_v1.json',
 'stage16_summary':REV/'00_integrity_audit'/'audit_summary_v1.json',
 'stage17_manifest':MAN/'stage17_evaluation_manifest_v1.json',
 'stage17_metrics':REV/'01_self_verification_baseline'/'stage17_final_test_metrics_v1.csv',
}
PRED=OUT/'stage18_ablation_predictions_v2.parquet'
FIT_AUDIT=OUT/'stage18_fit_audit_v2.json'
PRED_MAN=MAN/'stage18_prediction_manifest_v2.json'
METRICS=OUT/'stage18_final_test_metrics_v2.csv'
DIFFS=OUT/'stage18_ablation_effects_v2.csv'
BOOT=OUT/'stage18_paired_brier_bootstrap_v2.csv'
SELECT=OUT/'stage18_selective_risk_v2.csv'
REPORT=OUT/'stage18_run_report_v2.md'
EVAL_MAN=MAN/'stage18_evaluation_manifest_v2.json'
VARIANT_MAN=MAN/'stage18_variant_spec_v2.json'
HEAT_PNG=OUT/'stage18_delta_brier_heatmap_v2.png'
HEAT_PDF=OUT/'stage18_delta_brier_heatmap_v2.pdf'
HEAT_CSV=OUT/'stage18_heatmap_matrix_v2.csv'
SUMMARY_PNG=OUT/'stage18_reviewer_summary_heatmap_v2.png'
SUMMARY_PDF=OUT/'stage18_reviewer_summary_heatmap_v2.pdf'
SUMMARY_CSV=OUT/'stage18_reviewer_summary_matrix_v2.csv'
print('Environment:',platform.python_version(),'sklearn',sklearn.__version__,'xgboost',xgboost.__version__)
print('Inputs:',ROOT)
print('Outputs:',OUT)

In [ ]:
# 2 — İmmutable dosya işlemleri (eski aşamaları değiştirmez)
def require(ok,msg):
    if not bool(ok):raise RuntimeError(msg)
def digest(path):
    path=Path(path)
    require(path.is_file(),f'Girdi dosyası bulunamadı: {path}')
    h=hashlib.sha256()
    with path.open('rb') as fp:
        for chunk in iter(lambda:fp.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def readjson(path):return json.loads(Path(path).read_text(encoding='utf-8'))
def jsonbytes(obj):return (json.dumps(obj,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)+'\n').encode('utf-8')
def immutable_bytes(path,payload):
    path=Path(path);require(path.parent==OUT or path.parent==MAN,'Yalnız yeni revision klasörüne yazılabilir')
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        require(path.read_bytes()==payload,f'DOSYA ÇAKIŞMASI {path}; v2 oluşturulmalı (eski çıktı değiştirilemez).')
        print('UNCHANGED:',path.name)
    else:
        with path.open('xb') as fp:fp.write(payload)
        print('CREATED:',path.name)
    return digest(path)
def freeze_json(path,obj):return immutable_bytes(path,jsonbytes(obj))
def freeze_csv(path,frame):
    return immutable_bytes(path,frame.to_csv(index=False,float_format='%.15g',lineterminator='\n').encode('utf-8'))
def freeze_parquet(path,frame):
    path=Path(path);require(path.parent==OUT,'Geçersiz yazma yolu')
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        prev=pd.read_parquet(path)
        pd.testing.assert_frame_equal(prev.reset_index(drop=True),frame.reset_index(drop=True),check_exact=True)
        print('UNCHANGED:',path.name)
    else:
        temp=path.with_name(path.name+'.tmp.parquet')
        require(not temp.exists(),f'Geçici dosya var, önce incele: {temp}')
        frame.to_parquet(temp,index=False,engine='pyarrow')
        os.replace(temp,path)
        print('CREATED:',path.name)
    return digest(path)

def freeze_figure(path,fig,fmt,metadata=None):
    """Preserve previously generated figures instead of overwriting them."""
    path=Path(path)
    require(path.parent==OUT,'Geçersiz figure output yolu')
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        print('UNCHANGED (existing figure):',path.name)
        return digest(path)
    buf=BytesIO()
    fig.savefig(buf,format=fmt,dpi=300,bbox_inches='tight',
                facecolor='white',metadata=metadata)
    return immutable_bytes(path,buf.getvalue())


In [ ]:
# 3 — Stage 16 ve 17 başarı kontrolü; Stage12/14 orijinal hiperparametreleri sabitle
for name,path in P.items():require(path.is_file(),f'Eksik {name}: {path}')
s16=readjson(P['stage16_summary']);require(s16.get('status')=='PASS' and not s16.get('required_unresolved_checks'),'Stage16 denetimi PASS değil')
for label,expected in s16['input_sha256'].items():
    if label in {'dev_extended','dev_manifest','dev_schema','test_core','test_manifest'}:
        require(digest(P[label])==expected,f'Stage16 kaynak değişmiş: {label}')
    elif label=='frozen_predictions':
        require(digest(P['stage14_predictions'])==expected,'Stage14 predictions değişmiş')

mdev=readjson(P['dev_manifest']);mtest=readjson(P['test_manifest'])
pm=readjson(P['stage14_pred_manifest']);p12=readjson(P['stage12_protocol'])
s17=readjson(P['stage17_manifest'])
require(s17.get('status')=='PASS','Stage17 değerlendirme manifesti PASS değil')
require(s17.get('source_stage14_predictions_sha256')==digest(P['stage14_predictions']),'Stage17 frozen input SHA farklı')
require(s17.get('outputs_sha256',{}).get('metrics')==digest(P['stage17_metrics']),'Stage17 metrik çıktısı değişmiş')
require(pm.get('status')=='PREDICTIONS_FROZEN_BEFORE_TEST_LABEL_EVALUATION','Stage14 freeze status farklı')
require(pm['predictions_sha256']==digest(P['stage14_predictions']),'Stage14 kaynak tahminler değişmiş')
require(pm.get('training_n_per_model')==970 and pm.get('n_independent_test_questions')==250,'Stage14 sample sözleşmesi farklı')
require(mdev['core_features']==CORE and mtest['core_features']==CORE and pm['core_features_ordered']==CORE,'CORE feature sözleşmesi değişmiş')
require(digest(P['dev_extended'])==mdev['outputs']['extended_model_matrix_sha256'],'Dev feature SHA farklı')
require(digest(P['test_core'])==mtest['outputs']['core_model_matrix_sha256'],'Final feature SHA farklı')
require(digest(P['stage12_protocol'])==s16.get('input_sha256',{}).get('stage12_protocol',digest(P['stage12_protocol'])),'Stage12 protokol SHA')
require(p12['feature_names_ordered']==CORE and p12['target_column']==TARGET,'Stage12 feature/target kontratı')
require(int(p12['evaluation']['secondary_ece']['n_bins'])==ECE_BINS,'ECE bins değişmiş')

log_spec=p12['logistic_core'];xgb_spec=p12['xgb_core']
require(log_spec['features']==CORE and xgb_spec['features']==CORE,'Protokol CORE kontratı')
LOG_PARAMS=dict(log_spec['params'])
require(float(LOG_PARAMS['C'])==1.0 and LOG_PARAMS['solver']=='lbfgs' and int(LOG_PARAMS['max_iter'])==3000 and int(LOG_PARAMS['random_state'])==17,'Logistic sözleşmesi farklı')
XGB_SHARED=dict(xgb_spec['shared_parameters'])
# Stage14 dev-only inner CV ile her üç model için seçilmiş orijinal parametreleri kullan.
audit14=readjson(P['stage14_fit_audit'])
require(isinstance(audit14,list) and len(audit14)==3,'Stage14 fit audit formatı farklı')
SELECTED={record['model_id']:{'max_depth':int(record['xgb_selected_max_depth']),
                              'n_estimators':int(record['xgb_selected_n_estimators'])} for record in audit14}
require(all(v=={'max_depth':2,'n_estimators':120} for v in SELECTED.values()),'Stage14 XGBoost parametreleri beklenen 2/120 değil')
require(all(dict(v) in list(xgb_spec['hyperparameter_candidates']) for v in SELECTED.values()),'Stage14 seçimi protokol adaylarının dışında')
print('STAGES 12/14/16/17 integrity: PASS')
# Kalibratörleri eğitmeden önce altküme planını sabitle (test etiketlerinden bağımsız).
variant_plan={
 'stage':'18_comprehensive_feature_ablation','revision_version':'v2',
 'status':'POST_REVIEW_EXPLORATORY_PLAN_FIXED_BEFORE_NEW_PREDICTIONS',
 'core_features':CORE,'internal_features':INTERNAL,'behavioral_features':BEHAVIORAL,
 'internal_length_note':'output_content_token_count is output metadata, not token probability',
 'behavioral_verification_note':'verification_p_correct uses next-token probabilities; behavioral grouping does not imply black-box-only access',
 'reviewer_required_variants':['minus_consistency','minus_unknown','minus_both'],
 'feature_variants':FEATURES,'variant_labels':VARIANT_LABELS,
 'n_models':3,'n_calibrators_per_variant':2,'n_new_fits':N_FITS,
 'test_labels_used_for_feature_selection':False,
 'comparison':'delta_brier_variant_minus_frozen_full_core',
 'interpretation':'descriptive_post_review_exploratory_unadjusted',
 'bootstrap':{'B':BOOT_B,'seed':BOOT_SEED}}
variant_plan_sha=freeze_json(VARIANT_MAN,variant_plan)
print('Feature families:',{'internal':INTERNAL,'behavioral':BEHAVIORAL})
print('Variants:',{k:len(v) for k,v in FEATURES.items()})
print('Fixed v2 analysis plan SHA:',variant_plan_sha)
print('Locked XGB:',SELECTED)

In [ ]:
# 4 — Veri yükleme ve katı leakage kontrolleri
# TEST ETİKETLERİ BİLİNÇLİ OLARAK OKUNMUYOR.
cols_dev=KEY+['exclude_from_final_test','used_in_model_selection',TARGET]+CORE
full_dev=pd.read_parquet(P['dev_extended'],columns=cols_dev)
require(len(full_dev)==3000 and not full_dev.duplicated(KEY).any(),'Development satır/PK yanlış')
train=full_dev.loc[full_dev['exclude_from_final_test'].astype(int).eq(0)].copy()
require(len(train)==2910 and train.groupby('model_id').size().eq(970).all(),'970 Clean-Dev/model değil')
require(not train['used_in_model_selection'].astype(bool).any(),'Stage03 verisi eğitime karışmış')
require(train[TARGET].notna().all() and train[TARGET].isin([0,1]).all(),'Dev etiketleri 0/1 değil')

frozen=pd.read_parquet(P['stage14_predictions'])
require(len(frozen)==3000 and not frozen.duplicated(KEY+['method']).any(),'Stage14 frozen prediction contract')
require(frozen.groupby(['model_id','method']).size().eq(250).all(),'Frozen 250/model/method değil')
require(set(frozen['method'])=={'train_prevalence','raw_verification','logistic_core','xgb_core'},'Stage14 yöntemleri')
test_ids=set(frozen['question_id'].astype(str))
require(len(test_ids)==250 and set(train['question_id'].astype(str)).isdisjoint(test_ids),'Train/test ID sızıntısı')

test_all=pd.read_parquet(P['test_core'],columns=KEY+CORE)  # TARGET YÜKLENMEZ
test=test_all.loc[test_all['question_id'].astype(str).isin(test_ids)].copy()
require(len(test_all)==3750 and len(test)==750 and not test.duplicated(KEY).any(),'Test matrisi adet/key hatası')
models=sorted(train['model_id'].unique())
require(set(models)==set(frozen['model_id'])==set(test['model_id'])==set(SELECTED),'Model ID uyuşmazlığı')
require(test.groupby('model_id').size().eq(250).all(),'250 test örneği/model değil')
require(set(map(tuple,test[KEY].to_numpy()))==set(map(tuple,frozen[KEY].drop_duplicates().to_numpy())),'Frozen/model pairs farklı')
for name,frame in [('dev',train),('test',test)]:
    for feature in CORE:
        array=frame[feature].to_numpy(dtype=float)
        require(np.isfinite(array).all(),f'NaN/inf: {name}:{feature}')
    for feature in ['self_confidence_scaled','verification_p_correct','self_consistency','unknown_response']:
        require(frame[feature].between(0,1).all(),f'Feature sınır dışı: {name}:{feature}')
require(TARGET not in sum([list(v) for v in FEATURES.values()],[]),'Etiket predictor olmuş')
print('DEV:',len(train),'| TEST features:',len(test),'| Test gold henüz açılmadı.')

In [ ]:
# 5 — 11 altküme için yeni kalibratörler: yalnız 970 Clean-Dev ile fit, 250 bağımsız soruda predict
# Yeni tahminler ilk kez oluşturulurken test doğruluk etiketleri HİÇ OKUNMAZ.
if PRED.exists() or PRED_MAN.exists():
    require(PRED.exists() and PRED_MAN.exists() and FIT_AUDIT.exists(),'Kısmi Stage18 tahmin kaydı var')
    pp=readjson(PRED_MAN)
    require(pp.get('status')=='POST_REVIEW_EXPLORATORY_PREDICTIONS_FROZEN','Stage18 prediction manifest status')
    require(pp['new_predictions_sha256']==digest(PRED),'Stage18 tahmin SHA farklı')
    require(pp['stage14_predictions_sha256']==digest(P['stage14_predictions']),'Stage14 kaynak değişmiş')
    require(pp['dev_extended_sha256']==digest(P['dev_extended']),'Dev matrisi değişmiş')
    require(pp['test_core_sha256']==digest(P['test_core']),'Test matrisi değişmiş')
    require(pp['fit_audit_sha256']==digest(FIT_AUDIT),'Stage18 fit audit değişmiş')
    require(pp['variant_spec_sha256']==digest(VARIANT_MAN),'Stage18 variant spec değişmiş')
    newly=pd.read_parquet(PRED)
    print('PREVIOUSLY FROZEN STAGE18 PREDICTIONS — reused, no fitting')
else:
    out_rows=[];fit_details=[]
    for mid in models:
        dev_m=train.loc[train['model_id'].eq(mid)].sort_values('question_id',kind='mergesort').reset_index(drop=True)
        test_m=test.loc[test['model_id'].eq(mid)].sort_values('question_id',kind='mergesort').reset_index(drop=True)
        y=dev_m[TARGET].to_numpy(dtype=int)
        require(len(dev_m)==970 and len(test_m)==250 and len(np.unique(y))==2,'Train/test model counts')
        for variant,features in FEATURES.items():
            x=dev_m[features].to_numpy(dtype=float)
            xt=test_m[features].to_numpy(dtype=float)
            logistic=Pipeline([
                ('scaler',StandardScaler()),
                ('logistic',LogisticRegression(C=float(LOG_PARAMS['C']),solver=LOG_PARAMS['solver'],
                    max_iter=int(LOG_PARAMS['max_iter']),random_state=int(LOG_PARAMS['random_state'])))
            ])
            logistic.fit(x,y)
            xgb=XGBClassifier(**XGB_SHARED,**SELECTED[mid])
            xgb.fit(x.astype(np.float32),y,verbose=False)
            pairs=[(f'logistic_{variant}',logistic.predict_proba(xt)[:,1]),
                   (f'xgb_{variant}',xgb.predict_proba(xt.astype(np.float32))[:,1])]
            for method,probs in pairs:
                require(len(probs)==250 and np.isfinite(probs).all() and ((probs>=0)&(probs<=1)).all(),f'Probability range: {mid}/{method}')
                for question_id,p in zip(test_m['question_id'].astype(str),probs):
                    out_rows.append({'model_id':mid,'question_id':question_id,'method':method,'p_correct':float(p)})
            fit_details.append({'model_id':mid,'subset':variant,'feature_count':len(features),
                'features':features,'n_train':len(dev_m),'n_test':len(test_m),
                'positive_dev':int(y.sum()),'logistic_C':float(LOG_PARAMS['C']),
                'xgb_max_depth':int(SELECTED[mid]['max_depth']),
                'xgb_n_estimators':int(SELECTED[mid]['n_estimators'])})
        print('FITTED:',mid,f'{2*N_VARIANTS} subset calibrators; no LLM inference')
    newly=pd.DataFrame(out_rows).sort_values(['model_id','method','question_id'],kind='mergesort').reset_index(drop=True)
    require(len(newly)==N_NEW_PREDICTIONS and not newly.duplicated(KEY+['method']).any(),'Yeni ablation tahmin sayısı/PK')
    pred_sha=freeze_parquet(PRED,newly)
    fit_audit={'stage':'18_feature_ablation','status':'POST_REVIEW_EXPLORATORY',
       'feature_sets':FEATURES,'core_features':CORE,'internal_features':INTERNAL,'behavioral_features':BEHAVIORAL,'per_fit':fit_details,
       'logistic_params':LOG_PARAMS,'xgb_shared_params':XGB_SHARED,'xgb_selected_params_by_model':SELECTED,
       'n_new_fits':N_FITS,'llm_inference_calls':0,
       'test_labels_used_in_fit_or_selection':False,
       'variant_plan_sha256':digest(VARIANT_MAN),
       'environment':{'python':platform.python_version(),'sklearn':sklearn.__version__,'xgboost':xgboost.__version__}}
    fit_sha=freeze_json(FIT_AUDIT,fit_audit)
    pred_manifest={'stage':'18_feature_ablation_prediction','revision_version':'v2',
       'status':'POST_REVIEW_EXPLORATORY_PREDICTIONS_FROZEN','methods':METHODS,
       'n_new_predictions':N_NEW_PREDICTIONS,'n_test_questions_per_model':250,'n_clean_dev_per_model':970,
       'new_predictions_sha256':pred_sha,'variant_spec_sha256':digest(VARIANT_MAN),'fit_audit_sha256':fit_sha,
       'stage14_predictions_sha256':digest(P['stage14_predictions']),
       'dev_extended_sha256':digest(P['dev_extended']),'test_core_sha256':digest(P['test_core']),
       'stage16_summary_sha256':digest(P['stage16_summary']),
       'stage17_evaluation_manifest_sha256':digest(P['stage17_manifest']),
       'stage12_protocol_sha256':digest(P['stage12_protocol']),
       'pre_registered_with_original_stage12':False,'test_labels_used_in_fit_or_selection':False}
    freeze_json(PRED_MAN,pred_manifest)
require(len(newly)==N_NEW_PREDICTIONS and set(newly['method'])==set(METHODS),'New predictions contract')
require(newly.groupby(['model_id','method']).size().eq(250).all(),'250 model/variant')
require(not newly.duplicated(KEY+['method']).any() and newly['p_correct'].between(0,1).all(),'Invalid new predictions')
print('FROZEN new Stage18 predictions SHA:',digest(PRED))

## Tahmin dondurma / test etiketi bariyeri

Bu hücreye kadar, test dosyasından `manual_is_correct` sütunu **okunmaz**; yeni tahminler ve özellik varyant planı SHA-256 manifestleriyle dondurulur. Aşağıdaki hücre sadece sabit 250 sorunun etiketleriyle sonuçları hesaplar.

**Δ Brier = Brier(özellik altkümesi) − Brier(8 CORE).** Pozitif sonuçta Full CORE daha iyi; negatif sonuçta daraltılmış özellik seti daha iyi. Heatmap de **aynı işareti** kullanır. Tüm aralıklar hakem sonrası, keşifsel, çoklu karşılaştırma düzeltmesi yapılmamış eşleştirilmiş bootstrap aralıklarıdır.

In [ ]:
# 6 — Bağımsız test etiketleri yalnız değerlendirme için okunur
predm=readjson(PRED_MAN)
require(predm['new_predictions_sha256']==digest(PRED),'Stage18 tahmin dosyası değiştirilmiş')
require(predm['stage14_predictions_sha256']==digest(P['stage14_predictions']),'Stage14 tahmin dosyası değiştirilmiş')

labels=pd.read_parquet(P['test_core'],columns=KEY+[TARGET])
labels=labels.loc[labels['question_id'].astype(str).isin(test_ids)].copy()
require(len(labels)==750 and not labels.duplicated(KEY).any(),'250 test × 3 model etiket doğrulaması')
require(labels[TARGET].notna().all() and labels[TARGET].isin([0,1]).all(),'Test etiketleri 0/1 değil')

original=frozen.loc[frozen['method'].isin(BASE_METHODS),KEY+['method','p_correct']].copy()
combined=pd.concat([original,newly[KEY+['method','p_correct']]],ignore_index=True)
require(len(combined)==750*N_METHODS and not combined.duplicated(KEY+['method']).any(),f'Tahmin birleşimi {N_METHODS} yöntem × 750')
joined=combined.merge(labels,on=KEY,how='left',validate='many_to_one',indicator=True)
require(joined['_merge'].eq('both').all(),'Prediction–label join uyuşmazlığı')
require(joined['p_correct'].between(0,1).all() and np.isfinite(joined['p_correct']).all(),'Güven değerleri geçersiz')
require(joined.groupby(['model_id','method']).size().eq(250).all(),'250 x 8 method/model gerekli')

def metrics_for(ids,y,p):
    ids=np.asarray(ids,dtype=str);y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
    require(len(y)==250 and len(np.unique(y))==2,'Metrics expects 250 test with both classes')
    binids=np.minimum((np.clip(p,0.,1.)*ECE_BINS).astype(int),ECE_BINS-1)
    ece=sum(float((binids==b).mean())*abs(float(y[binids==b].mean())-float(p[binids==b].mean()))
            for b in range(ECE_BINS) if (binids==b).any())
    safe=np.clip(p,EPS,1.-EPS)
    order=np.lexsort((ids,-p))
    curve=np.cumsum((1-y)[order])/np.arange(1,len(y)+1)
    return {'n':len(y),'prevalence':float(y.mean()),
     'brier':float(np.mean((p-y)**2)),
     'nll':float(-np.mean(y*np.log(safe)+(1-y)*np.log1p(-safe))),
     'ece':float(ece),'auroc':float(roc_auc_score(y,p)),
     'auprc':float(average_precision_score(y,p)),'aurc':float(np.mean(curve))},curve

rows=[];boots=[];selective=[];diffs=[]
COMPARISONS=[(f'{base}_core',f'{base}_{variant}',variant)
             for base in ('logistic','xgb') for variant in FEATURES]
for model_id,group in joined.groupby('model_id',sort=True):
    wide=group.pivot(index='question_id',columns='method',values='p_correct').sort_index()
    correct=group[['question_id',TARGET]].drop_duplicates('question_id').set_index('question_id').sort_index()
    require(wide.index.equals(correct.index) and len(wide)==250,'250 paired IDs mismatch')
    ids=wide.index.to_numpy(dtype=str);y=correct[TARGET].to_numpy(dtype=int)
    for method in BASE_METHODS+METHODS:
        m,curve=metrics_for(ids,y,wide[method].to_numpy(dtype=float))
        rows.append({'model_id':model_id,'method':method,**m})
        for coverage in (0.2,0.5,0.8,1.0):
            k=max(1,int(math.ceil(coverage*250)))
            selective.append({'model_id':model_id,'method':method,'coverage':coverage,'n_selected':k,'risk':float(curve[k-1])})
    # Aynı sorular üzerinde paired bootstrap; seed raporda kaydedilir
    sample=np.random.default_rng(BOOT_SEED).integers(0,250,size=(BOOT_B,250),endpoint=False)
    for reference,alternative,variant in COMPARISONS:
        pref=wide[reference].to_numpy(dtype=float)
        palt=wide[alternative].to_numpy(dtype=float)
        delta=(palt-y)**2-(pref-y)**2
        ci=np.quantile(delta[sample].mean(axis=1),[0.025,0.975])
        boots.append({'model_id':model_id,'reference':reference,'alternative':alternative,
          'variant':variant,'effect_brier_ablation_minus_full':float(delta.mean()),
          'ci95_low':float(ci[0]),'ci95_high':float(ci[1]),
          'bootstrap_B':BOOT_B,'seed':BOOT_SEED,'n_questions':250,
          'interval':'post_review_exploratory_paired_question_percentile_95_unadjusted'})
        diffs.append({'model_id':model_id,'method_family':reference.split('_')[0],
          'removed':variant,'full_brier':float(np.mean((pref-y)**2)),
          'ablated_brier':float(np.mean((palt-y)**2)),
          'delta_brier_ablation_minus_full':float(delta.mean()),
          'ci95_low':float(ci[0]),'ci95_high':float(ci[1])})

met=pd.DataFrame(rows).sort_values(['model_id','method']).reset_index(drop=True)
boot=pd.DataFrame(boots).sort_values(['model_id','reference','variant']).reset_index(drop=True)
effects=pd.DataFrame(diffs).sort_values(['model_id','method_family','removed']).reset_index(drop=True)
select_df=pd.DataFrame(selective).sort_values(['model_id','method','coverage']).reset_index(drop=True)
require(len(met)==3*N_METHODS and len(boot)==N_FITS and len(effects)==N_FITS and len(select_df)==3*N_METHODS*4,'Stage18 output row counts')

# Makalede yayımlanan orijinal CORE sonuçları birebir korunmuş mu?
orig=pd.read_csv(P['stage14_metrics'])
ref=orig.loc[orig['method'].isin(BASE_METHODS)]
matched=met.loc[met['method'].isin(BASE_METHODS)].merge(ref,on=['model_id','method'],validate='one_to_one',suffixes=('_now','_stage14'))
require(len(matched)==6,'Stage14 CORE matches 6 rows')
for m in ['n','prevalence','brier','nll','ece','auroc','auprc','aurc']:
    err=float(np.max(np.abs(matched[f'{m}_now']-matched[f'{m}_stage14'])))
    require(err<2e-9,f'Stage14 CORE reproduction failed: {m}, delta={err}')
print('ORIGINAL CORE RESULTS UNCHANGED: PASS')
print('STAGE18 FITS (reviewer-requested):')
print(met[['model_id','method','brier','ece','auroc','aurc']].to_string(index=False,float_format=lambda x:f'{x:.6f}'))
print('ABLATION EFFECTS: positive delta = feature removal worsens Brier')
print(effects[['model_id','method_family','removed','delta_brier_ablation_minus_full','ci95_low','ci95_high']].to_string(index=False,float_format=lambda x:f'{x:.5f}'))

In [ ]:
# 7 — Özet Δ Brier heatmap (yayına uygun PNG / PDF) + sayısal matrisi
# Tek ısı haritasında 11 altküme × 6 model/kalibratör. Ortada sıfır.
MODEL_SHORT={
 'Qwen/Qwen3-4B':'Qwen3-4B',
 'google/gemma-3-4b-it':'Gemma-3-4B',
 'microsoft/Phi-4-mini-instruct':'Phi-4-mini',
}
DISPLAY_COLS=[(mid,family) for mid in models for family in ('logistic','xgb')]
rows=[]
for variant in VARIANT_ORDER:
    record={'variant':variant,'label':VARIANT_LABELS[variant],
            'n_features':len(FEATURES[variant])}
    for mid,family in DISPLAY_COLS:
        sel=effects.loc[effects['model_id'].eq(mid) &
                        effects['method_family'].eq(family) &
                        effects['removed'].eq(variant)]
        require(len(sel)==1,f'Heatmap duplicate/missing {mid} {family} {variant}')
        col=f'{MODEL_SHORT[mid]} | {"LR" if family=="logistic" else "XGB"}'
        record[col]=float(sel.iloc[0]['delta_brier_ablation_minus_full'])
    rows.append(record)
heat_table=pd.DataFrame(rows)
heat_sha=freeze_csv(HEAT_CSV,heat_table)
colnames=[f'{MODEL_SHORT[mid]} | {"LR" if family=="logistic" else "XGB"}'
          for mid,family in DISPLAY_COLS]
z=heat_table[colnames].to_numpy(dtype=float)
require(z.shape==(N_VARIANTS,6) and np.isfinite(z).all(),'Heatmap shape or values')
lim=max(0.005,float(np.max(np.abs(z))))
fig,ax=plt.subplots(figsize=(11.4,7.2),dpi=130)
img=ax.imshow(z,aspect='auto',interpolation='nearest',
              cmap='RdBu_r',norm=TwoSlopeNorm(vcenter=0,vmin=-lim,vmax=lim))
ax.set_xticks(np.arange(len(colnames)),colnames,rotation=27,ha='right',fontsize=10)
ax.set_yticks(np.arange(N_VARIANTS),heat_table['label'].tolist(),fontsize=10)
ax.set_title('Effect of feature subsets on Brier score',fontsize=15,weight='bold',pad=20)
ax.set_xlabel('LLM and calibrator',fontsize=11)
ax.set_ylabel('Feature set relative to Full CORE',fontsize=11)
for i in range(N_VARIANTS):
    for j in range(len(colnames)):
        value=float(z[i,j])
        text_color='white' if abs(value)>=lim*0.60 else '#252525'
        ax.text(j,i,f'{value:+.3f}',ha='center',va='center',
                fontsize=9,color=text_color,weight='medium')
# reviewer rows after LOFO: mark as reviewers without arbitrarily ranking significance
for idx in (VARIANT_ORDER.index('minus_consistency'),VARIANT_ORDER.index('minus_unknown')):
    ax.get_yticklabels()[idx].set_fontweight('bold')
ax.axhline(7.5,color='#555555',linewidth=1.0)
ax.axhline(8.5,color='#555555',linewidth=1.0)
ax.tick_params(axis='both',length=0)
for spine in ax.spines.values():spine.set_visible(False)
cbar=fig.colorbar(img,ax=ax,pad=0.015,fraction=0.038)
cbar.set_label('Δ Brier = subset − Full CORE\n(positive: Full CORE better)',fontsize=10)
fig.text(0.5,0.015,
    'Post-review exploratory; unadjusted descriptive comparisons. Bold: reviewer-requested leave-one-out tests.',
    ha='center',va='bottom',fontsize=9)
fig.subplots_adjust(left=0.285,right=0.92,bottom=0.17,top=0.88)
png_sha=freeze_figure(HEAT_PNG,fig,'png',metadata={'Software':'MIDI2026 Stage18 v2'})
pdf_sha=freeze_figure(HEAT_PDF,fig,'pdf',metadata={'Creator':'MIDI2026 Stage18 v2','CreationDate':None,'ModDate':None})
plt.show()
plt.close(fig)
print('Saved heatmap CSV:',HEAT_CSV.name,'SHA:',heat_sha)
print('Saved figure PNG:',HEAT_PNG.name,'SHA:',png_sha)
print('Saved figure PDF:',HEAT_PDF.name,'SHA:',pdf_sha)

# Yayın için kompakt özet: 3 hakem odaklı karşılaştırma + 2 aile karşılaştırması.
SUMMARY_VARIANTS=['minus_consistency','minus_unknown','minus_both',
                  'internal_only','behavioral_only']
summary_table=heat_table.loc[heat_table['variant'].isin(SUMMARY_VARIANTS)].copy()
summary_table['order']=summary_table['variant'].map({v:i for i,v in enumerate(SUMMARY_VARIANTS)})
summary_table=summary_table.sort_values('order').drop(columns='order').reset_index(drop=True)
require(len(summary_table)==5,'Missing summary variants')
summary_sha=freeze_csv(SUMMARY_CSV,summary_table)
summary_z=summary_table[colnames].to_numpy(dtype=float)
fig_small,ax_small=plt.subplots(figsize=(8.6,4.35),dpi=140)
img_small=ax_small.imshow(summary_z,aspect='auto',interpolation='nearest',
                          cmap='RdBu_r',norm=TwoSlopeNorm(vcenter=0,vmin=-lim,vmax=lim))
short_labels=[f'{MODEL_SHORT[mid]}\n{"LR" if family=="logistic" else "XGB"}'
              for mid,family in DISPLAY_COLS]
ax_small.set_xticks(np.arange(6),short_labels,fontsize=9)
ax_small.set_yticks(np.arange(5),summary_table['label'].tolist(),fontsize=9)
ax_small.set_title('Reviewer-focused feature ablation (Δ Brier)',fontsize=13,weight='bold',pad=14)
for i in range(5):
    for j in range(6):
        val=float(summary_z[i,j])
        ax_small.text(j,i,f'{val:+.3f}',ha='center',va='center',fontsize=8.5,
                      color='white' if abs(val)>=lim*0.60 else '#252525')
ax_small.axhline(2.5,color='#555555',linewidth=1.0)
ax_small.tick_params(axis='both',length=0)
for spine in ax_small.spines.values():spine.set_visible(False)
cb_small=fig_small.colorbar(img_small,ax=ax_small,pad=0.014,fraction=0.046)
cb_small.set_label('Δ Brier: subset − Full CORE',fontsize=9)
fig_small.text(0.5,0.015,'Post-review exploratory; positive Δ favors Full CORE.',
               ha='center',va='bottom',fontsize=8.5)
fig_small.subplots_adjust(left=0.245,right=0.915,bottom=0.20,top=0.86)
summary_png_sha=freeze_figure(SUMMARY_PNG,fig_small,'png',
                              metadata={'Software':'MIDI2026 Stage18 v2'})
summary_pdf_sha=freeze_figure(SUMMARY_PDF,fig_small,'pdf',
                              metadata={'Creator':'MIDI2026 Stage18 v2',
                                        'CreationDate':None,'ModDate':None})
plt.show()
plt.close(fig_small)
print('Saved publication summary figure:',SUMMARY_PDF.name,'SHA:',summary_pdf_sha)


In [ ]:
# 8 — Sonuç dosyalarını kaydet, rapor oluştur ve v2 audit manifestini dondur
met_sha=freeze_csv(METRICS,met)
boot_sha=freeze_csv(BOOT,boot)
effect_sha=freeze_csv(DIFFS,effects)
select_sha=freeze_csv(SELECT,select_df)

lines=['# MIDI2026 Stage 18 v2 — Comprehensive CORE Feature Ablation', '',
  '**Status:** PASS','',
  '**Nature:** post-review exploratory; original Stage12/14 primary result remains unchanged.','',
  '- 3 models × 970 Clean-Dev train / 250 frozen final-test samples',
  '- 8 leave-one-out (including reviewer-required consistency and unknown), 1 minus-both, 2 group-only configurations',
  f'- {N_VARIANTS} subsets × 2 calibrators × 3 models = {N_FITS} additional small calibrator fits',
  f'- {N_NEW_PREDICTIONS} new predictions, zero LLM inference',
  '- Full CORE predictions reused from Stage14; XGBoost parameters fixed from development',
  '- Brier, ECE, AUROC, AURC, NLL and selective prediction computed for every subset',
  '- B=4000 paired-question percentile bootstrap CIs, unadjusted and descriptive',
  '', '## Feature family definitions',
  '- Internal (+ output metadata): '+', '.join(INTERNAL),
  '- Behavioral (+ derived unknown marker): '+', '.join(BEHAVIORAL),
  '- Output token count is metadata, not a model-internal token probability.',
  '- Behavioral verification_p_correct depends on next-token probabilities and is not strictly black-box.',
  '', '## Reviewer-required Brier ablation results', '',
  '| Model | Calibrator | Full CORE | − Consistency | − Unknown | − Both |',
  '|---|---|---:|---:|---:|---:|']
for mid in models:
    p=met.loc[met['model_id'].eq(mid)].set_index('method')
    for family in ('logistic','xgb'):
        methods=[family+'_core',family+'_minus_consistency',family+'_minus_unknown',family+'_minus_both']
        lines.append('| '+mid+' | '+family+' | '+' | '.join(f'{p.at[m,"brier"]:.4f}' for m in methods)+' |')
lines += ['', '## Internal / behavioral group-only comparison', '',
 '| Model | Calibrator | Full CORE | Internal-only | Behavioral-only |',
 '|---|---|---:|---:|---:|']
for mid in models:
    p=met.loc[met['model_id'].eq(mid)].set_index('method')
    for family in ('logistic','xgb'):
        methods=[family+'_core',family+'_internal_only',family+'_behavioral_only']
        lines.append('| '+mid+' | '+family+' | '+' | '.join(f'{p.at[m,"brier"]:.4f}' for m in methods)+' |')
lines += ['', '## Interpretation and limitations',
  '- Δ Brier = Brier(subset) − Brier(full CORE). Positive Δ favors full CORE; negative favors subset.',
  '- The effect is conditional on model refitting with the remaining features; do not call it causal importance.',
  '- The test results were known before reviewer-requested analyses were planned: all intervals are exploratory / descriptive.',
  '- With 66 comparisons, no unadjusted interval should be presented as multiplicity-controlled evidence.',
  '- Removing self-consistency can avoid five extra resampled generations **only if no other retained feature uses them**.',
  '- No token counts, latency, compute or energy measurements are reported or implied.',
  '', '## Outputs',
  '- stage18_ablation_predictions_v2.parquet',
  '- stage18_variant_spec_v2.json',
  '- stage18_fit_audit_v2.json',
  '- stage18_final_test_metrics_v2.csv',
  '- stage18_ablation_effects_v2.csv',
  '- stage18_paired_brier_bootstrap_v2.csv',
  '- stage18_selective_risk_v2.csv',
  '- stage18_heatmap_matrix_v2.csv',
  '- stage18_delta_brier_heatmap_v2.png/.pdf (full)',
  '- stage18_reviewer_summary_heatmap_v2.png/.pdf (compact paper candidate)',
  '- stage18_reviewer_summary_matrix_v2.csv',
  '- stage18_run_report_v2.md',
  '- stage18_evaluation_manifest_v2.json', '']
report_sha=immutable_bytes(REPORT,('\n'.join(lines)+'\n').encode('utf-8'))
eval_manifest={
 'stage':'18_midi2026_comprehensive_feature_ablation','revision_version':'v2','status':'PASS',
 'study_status':'post_review_exploratory',
 'n_models':3,'n_train_per_model':970,'n_test_questions_per_model':250,
 'feature_groups':{'internal_plus_metadata':INTERNAL,'behavioral_plus_response_derived':BEHAVIORAL},
 'n_feature_subsets':N_VARIANTS,'new_calibrator_fits':N_FITS,
 'llm_calls':0,'n_new_predictions':N_NEW_PREDICTIONS,
 'new_prediction_sha256':digest(PRED),
 'variant_spec_sha256':digest(VARIANT_MAN),
 'prediction_manifest_sha256':digest(PRED_MAN),
 'stage14_predictions_sha256':digest(P['stage14_predictions']),
 'stage14_metrics_sha256':digest(P['stage14_metrics']),
 'stage16_summary_sha256':digest(P['stage16_summary']),
 'stage17_evaluation_manifest_sha256':digest(P['stage17_manifest']),
 'stage12_protocol_sha256':digest(P['stage12_protocol']),
 'outputs_sha256':{
   'metrics':met_sha,'paired_bootstrap':boot_sha,
   'effects':effect_sha,'selective_risk':select_sha,'heatmap_data':heat_sha,
   'heatmap_png':png_sha,'heatmap_pdf':pdf_sha,
   'summary_data':summary_sha,'summary_png':summary_png_sha,
   'summary_pdf':summary_pdf_sha,
   'report':report_sha,'fit_audit':digest(FIT_AUDIT)
 },
 'bootstrap':{'B':BOOT_B,'seed':BOOT_SEED,'type':'paired_question_percentile_95',
              'multiple_comparison_adjustment':False},
 'original_stage14_primary_comparison_modified':False,
 'test_labels_used_in_training':False
}
freeze_json(EVAL_MAN,eval_manifest)
print('\nSTAGE 18 V2: PASS — 11 subsets × 2 calibrators × 3 models')
print('Please share these files for analysis:')
for p in (METRICS,DIFFS,BOOT,FIT_AUDIT,REPORT,EVAL_MAN,HEAT_PNG,HEAT_CSV,SUMMARY_PNG,SUMMARY_CSV):
    print(' -',p)
